<a href="https://colab.research.google.com/github/marcohuertas/AI-agents-projects/blob/main/creating_biomodels_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PURPOSE

This notebook contains the code used to create a dataset of curated model descriptions obtained from Biomodels (www.biomodels.org). The dataset focuses on models consisting of systems of ordinary differential equations and that have been used for oncology projects.

The final dataset is uploaded to Hugging Face for further use.

This dataset has been created as part of a project on AI Agents.

## NOTE: Issues and possible fixed for accessing Biomodels from Colab
In many instances there are queries that return the status code 403 from the Restful API. According to Gemini a possible path to solve this problem is the following:

 1. The 10-Second Workaround (If blocked) If you run your code and instantly get a 403, do not waste time rewriting headers. Use the fast VM rotation trick:
    - Click Runtime in the top menu.Click Disconnect and delete runtime.Re-run your notebook.

 2. Otherwise, try running the following code
  ```
  import os
  def use_backup_route(enabled=True):
      if enabled:
          # Routes traffic through a free, public HTTP proxy
          # (Replace with a reliable free or premium proxy URL if needed)
          proxy_url = "http://pubproxy.com"
          
          # Example of setting a static free proxy
          os.environ['HTTP_PROXY'] = 'http://45.77.56.114:8080'
          os.environ['HTTPS_PROXY'] = 'http://45.77.56.114:8080'
          print("🔄 Traffic successfully rerouted away from Google Cloud.")
      else:
          os.environ.pop('HTTP_PROXY', None)
          os.environ.pop('HTTPS_PROXY', None)
          print("➔ Using standard Google Cloud network routing.")

  # UNCOMMENT THE LINE BELOW ONLY IF YOU GET A 403 ERROR
  # use_backup_route(True)
  ```

# LOAD PACKAGES

In [ ]:
import os
from datetime import datetime
import io
import requests
import pandas as pd
import zipfile
from collections import defaultdict
from time import sleep
from tqdm import tqdm

# HF API
# Make sure that you have a Hugging Face token.
from huggingface_hub import notebook_login
notebook_login()

from datasets import Dataset

Possible work around when queries to BioModels return 403 status too frequently

In [ ]:
# NOTE: Code provided by Google Gemini to deal with the isssue of accessing Biomodels.
# Does not always work as it times out.
def use_backup_route(enabled=True):
    if enabled:
        # Routes traffic through a free, public HTTP proxy
        # (Replace with a reliable free or premium proxy URL if needed)
        proxy_url = "http://pubproxy.com"

        # Example of setting a static free proxy
        os.environ['HTTP_PROXY'] = 'http://45.77.56.114:8080'
        os.environ['HTTPS_PROXY'] = 'http://45.77.56.114:8080'
        print("🔄 Traffic successfully rerouted away from Google Cloud.")
    else:
        os.environ.pop('HTTP_PROXY', None)
        os.environ.pop('HTTPS_PROXY', None)
        print("➔ Using standard Google Cloud network routing.")

# UNCOMMENT THE LINE BELOW ONLY IF YOU GET A 403 ERROR
# use_backup_route(True)

In [ ]:
use_backup_route(False)

# GET MODELS


In [ ]:
modelingapproach = 'modellingapproach:"ordinary differential equation model"'
modelformat = 'modelformat:"SBML"'
submitterkeywords = 'submitter_keywords:"*oncology"'
query = ' AND '.join(['*:*', modelingapproach, modelformat, submitterkeywords])
print(query)

In [ ]:
query_url = "https://biomodels.org/search"
query_parms = {
    "query" : query,
    "offset" : "0",
    "numResults" : "200",
    "sort" : "publication_year-desc",
    "format" : "json",
}
query_results = requests.get(query_url, params=query_parms)
print(query_results.status_code)

results = query_results.json()
print(results['matches'])

dfmodels = pd.DataFrame.from_records(results['models'])

In [ ]:
dfmodels.shape

The query has 120 matches and this first call pulled only 100. We will pull the remaining 20 by modifying the offset to 100 and run another query.
Also create a list of dataframes that will be concatenated later.

In [ ]:
df_list = [dfmodels,]

In [ ]:
query_url = "https://biomodels.org/search"
query_parms = {
    "query" : query,
    "offset" : "100",
    "numResults" : "100",
    "sort" : "publication_year-desc",
    "format" : "json",
}
query_results = requests.get(query_url, params=query_parms)
print(query_results.status_code)

results = query_results.json()
print(results['matches'])

dfmodels = pd.DataFrame.from_records(results['models'])

In [ ]:
dfmodels.shape

In [ ]:
# Append new dataframe
df_list.append(dfmodels)

In [ ]:
dfmodels = pd.concat(df_list, axis=0)
dfmodels.shape

In [ ]:
filename = "/content/biomodels.csv"
dfmodels.to_csv(filename, index=False)

# GET MODEL INFO
 - Get information about the type of publication id (either PMDI or DOI) that can be used to connect with the PubMed article.
 - Curation status
 - Synopsis
 - Description

In [ ]:
# Get model information and put it in a pandas DataFrame
model_id_list = dfmodels.id.tolist()
model_id_list[:5]

In [ ]:
API_URL = "https://biomodels.org"
output_format = "json"

model_files = []
for model_id in model_id_list:
  query = API_URL + "/" + model_id + "?format=" + output_format
  query_response = requests.get(query)

  if query_response.status_code==200:
      model_info = query_response.json()
      pubtype = model_info['publication']['type']
      synopsis = model_info['publication']['synopsis']
      accession = model_info['publication']['accession']
      description = model_info['description']
      curationstatus = model_info['curationStatus']

      if pubtype=='PubMed ID':
          idtype = 'pmid'

      if pubtype=='DOI':
          idtype = 'doi'

      filenames = []
      if 'main' in model_info['files']:
          for files in model_info['files']['main']:
            filenames.append(files['name'])

      model_files.append(
          {
              'id': model_id,
              'pubtype': pubtype,
              'accession': accession,
              'idtype': idtype,
              'synopsis': synopsis,
              'modelfiles': filenames,
              'description': description,
              'curationstatus': curationstatus,
          }
      )
  else:
      print(query_response.status_code)
      print(f'Error retrieven data for model id: {model_id}')

print("DONE")

In [ ]:
dfmodelinfo = pd.DataFrame.from_records(model_files)
dfmodelinfo.shape

In [ ]:
filename = "/content/biomodels_info.csv"
dfmodelinfo.to_csv(filename, index=False)

# GET MODEL ENTITIES
These are the names used to describe the species. There are two formats
 - entity: contains general names
 - entity_id: names used in the model

The format for these two quantities for each model is the following, both based on strings separated by '|':
- entity:
  - '\<entity#1\> | \<entity#2\> | \<entity#3\>'
- entity_id:
  - '\<entity_id#1\> | \<entity_id#2\> | \<entity_id#3\>'

Here is an example from the data:
- entity:

'[neoplastic cell; CL:0001201; CHEBI:144829]|[T cell; C122157]|[T cell]|[T cell; C137999]'

- entity_id

'[Antigen presenting tumour cells]|[Total lymphocyte count]|[Normal T cells]|[CAR T cells]'

These are connected as in this dictionary:
```
{
  '[neoplastic cell; CL:0001201; CHEBI:144829]': '[Antigen presenting tumour cells]',
  '[T cell; C122157]': '[Total lymphocyte count]',
  '[T cell]': '[Normal T cells]',
  '[T cell; C137999]': '[CAR T cells]'
 }
 ```

In [ ]:
def get_model_entities(model_id):
  API_URL = "https://biomodels.org"
  url = API_URL + "/parameterSearch/search"
  query_parms = {
      "query": model_id,
      "start": "0",
      "size": "100",
      "sort": "model:ascending",
      "format": "json",
  }
  query_results = requests.get(url, params=query_parms)

  # Dictionary with final unique entities and entity_ids
  entity_dict = defaultdict(str)
  entity_dict['id'] = model_id
  entity_dict['request_error']

  if query_results.status_code==200:
    results = query_results.json()
    # Check that all records were retrieved
    if results['recordsFiltered']==results['recordsTotal']:
      # print(f'Records retrieved: {results['recordsFiltered']} of {results['recordsTotal']}')
      entries = results['entries']

      entity_map_dict = defaultdict(list)
      for entry in entries:
        entity_id = entry['fields']['entity_id']
        entity = entry['fields']['entity']
        entity_map_dict[entity] = list(set(entity_map_dict[entity] + [entity_id]))

      entity_dict['entity'] = "|".join(list(entity_map_dict.keys()))
      entity_dict['entity_id'] = "|".join([f"[{u}]" for u in ["; ".join(u) for u in list(entity_map_dict.values())]])
    else:
      print(f'Records retrieved: {results['recordsFiltered']} of {results['recordsTotal']}')
      entity_dict['request_error'] = str(f'Records retrieved: {results['recordsFiltered']} of {results['recordsTotal']}')
  else:
    print(model_id, query_results.status_code)
    entity_dict['request_error'] = str(query_results.status_code)

  dfparms = pd.DataFrame.from_dict([entity_dict])
  return dfparms

In [ ]:
dflist = []
# for model_id in model_id_list:
for idx in tqdm(range(len(model_id_list))):
  model_id = model_id_list[idx]
  dflist.append(get_model_entities(model_id))
  sleep(1) # pace yourself, wait a second


In [ ]:
dfmodelentities = pd.concat(dflist).reset_index(drop=True)
dfmodelentities.head()

In [ ]:
# Check that no errors happened during the requests
dfissues = dfmodelentities[dfmodelentities.request_error!=""]
dfissues

In [ ]:
filename = "/content/biomodels_entities.csv"
dfmodelentities.to_csv(filename, index=False)

# GET MODEL SBML FILES
This code was generated by Google's Gemini

In [ ]:
# NOTE: Code provided by Google Gemini to download the model files into a pandas DataFrame

def download_bulk_to_dataframe(model_ids):
    if model_ids is None:
        return None

    API_URL = "https://biomodels.org"
    download_url: str = API_URL + "/search/download?models=" + model_ids
    headers = {"Content-Type": "application/zip"}

    # Request the data
    response = requests.get(download_url, headers=headers, allow_redirects=True)

    # Check if the request was successful
    response.raise_for_status()

    # Wrap the raw response bytes in an in-memory buffer
    zip_buffer = io.BytesIO(response.content)

    # Lists to store our data for the DataFrame
    model_id_list = []
    sbml_content_list = []

    # Unzip the file entirely in-memory
    with zipfile.ZipFile(zip_buffer) as z:
        for file_info in z.infolist():
            # Skip directories if any exist in the zip
            if file_info.is_dir():
                continue

            # Read the file content and decode it from bytes to a UTF-8 string
            with z.open(file_info) as file:
                sbml_text = file.read().decode("utf-8")

            # Extract a clean model ID from the filename (e.g., "BIOMD0000000001.xml" -> "BIOMD0000000001")
            model_id = os.path.splitext(file_info.filename)[0]

            model_id_list.append(model_id)
            sbml_content_list.append(sbml_text)

    # Create the pandas DataFrame
    df = pd.DataFrame({"model_id": model_id_list, "sbml_string": sbml_content_list})

    return df

It seems that one cannot process large lists of models, so here the list is split in half, which seems to work well.

In [ ]:
thismodelids = ",".join(model_id_list[:60])
dfsbmlfiles1 = download_bulk_to_dataframe(thismodelids)
print(dfsbmlfiles1.shape)

In [ ]:
thismodelids = ",".join(model_id_list[60:])
dfsbmlfiles2 = download_bulk_to_dataframe(thismodelids)
print(dfsbmlfiles1.shape)

In [ ]:
dfsbmlfiles = pd.concat([dfsbmlfiles1, dfsbmlfiles2])
dfsbmlfiles.rename(columns={"model_id": "id"}, inplace=True)
print(dfsbmlfiles.shape)

In [ ]:
filename = "/content/biomodels_sbmlfiles.csv"
dfsbmlfiles.to_csv(filename, index=False)

# PUTTING ALL TOGETHER

In [ ]:
df = pd.merge(
    left=dfmodels,
    right=dfmodelinfo,
    on="id"
)

df = pd.merge(
    left=df,
    right=dfmodelentities,
    on="id"
)

df = pd.merge(
    left=df,
    right=dfsbmlfiles,
    on="id"
)

df.shape

In [ ]:
filename = "/content/biomodels_odes_sbml_oncology.csv"
df.to_csv(filename, index=False)

# UPLOAD TO HUGGING FACE

In [ ]:
drop_columns = ["lastModified", "submissionDate", "submitter", "request_error"]
dffinal = df.drop(columns=drop_columns)
dffinal.head()

In [ ]:
hf_dataset = Dataset.from_pandas(dffinal, split='train', preserve_index=False)


In [ ]:
dataset_name = "biomodels-sbml-odemodels-oncology"
hf_dataset.push_to_hub(
    dataset_name,
    commit_description="Upload new dataset with ODE models used in oncology"
    )

# SCRAPS